In [1]:
from astropy.io import fits
from astropy import units as u
from spectral_cube import SpectralCube as sc
import warnings
from astropy.utils.exceptions import AstropyWarning
from tqdm.notebook import trange

# Suppress the specific Astropy warning
warnings.filterwarnings('ignore', message='No observer defined on WCS', category=AstropyWarning)

%matplotlib widget

import sys
print(sys.version)
print(sys.executable)

3.13.3 | packaged by conda-forge | (main, Apr 14 2025, 20:44:03) [GCC 13.3.0]
/home/firestar/.miniconda/bin/python


In [2]:
# Reproject the HI4PI cube to match the CRAFTS data

def reproject_cube(cube, target_header):
    reprojected_cube = cube.reproject(target_header, roundtrip_coords=False, parallel=True)
    return reprojected_cube

In [3]:
# Define the source and target cube patterns
source_pattern = "./HI4PI/HI4PI_RA{ra}_DEC-13_2.fits"
target_pattern = "./CRAFTS_RA{ra}_DEC-13_2_corrected_{sign}.fits"
output_pattern = "./HI4PI/HI4PI_RA{ra}_DEC-13_2_reproj_{sign}.fits"

# Define RA ranges and signs
ra_ranges = ["60_80", "80_100", "100_120", "120_140"]
signs = ["+", "-"]

# Enqueue all tasks
for i in trange(len(ra_ranges)):
    for j in range(len(signs)):
        ra = ra_ranges[i]
        sign = signs[j]

        source_file = source_pattern.format(ra=ra)
        target_file = target_pattern.format(ra=ra, sign=sign)
        output_file = output_pattern.format(ra=ra, sign=sign)

        # Open the source and target FITS files
        cube = sc.read(source_file)
        cube = cube.with_spectral_unit(u.km/u.s)

        target_hdul = fits.open(target_file)
        target_header = target_hdul[0].header

        reprojected_cube = reproject_cube(cube, target_header)

        reprojected_cube.write(output_file, format="fits", overwrite=True)
        print(f"Reprojected cube saved as {output_file}")
        del cube
        del reprojected_cube

  0%|          | 0/4 [00:00<?, ?it/s]

Reprojected cube saved as ./HI4PI/HI4PI_RA60_80_DEC-13_2_reproj_+.fits
Reprojected cube saved as ./HI4PI/HI4PI_RA60_80_DEC-13_2_reproj_-.fits
Reprojected cube saved as ./HI4PI/HI4PI_RA80_100_DEC-13_2_reproj_+.fits
Reprojected cube saved as ./HI4PI/HI4PI_RA80_100_DEC-13_2_reproj_-.fits
Reprojected cube saved as ./HI4PI/HI4PI_RA100_120_DEC-13_2_reproj_+.fits
Reprojected cube saved as ./HI4PI/HI4PI_RA100_120_DEC-13_2_reproj_-.fits
Reprojected cube saved as ./HI4PI/HI4PI_RA120_140_DEC-13_2_reproj_+.fits
Reprojected cube saved as ./HI4PI/HI4PI_RA120_140_DEC-13_2_reproj_-.fits
